# ETLegacy Core Infrastructure

ETLegacy writes Einstein Toolkit thorn metadata and C source files from NRPy registries. This notebook builds a small generation-only thorn and inspects the files that connect NRPy-generated kernels to Cactus.

[Index](../index.ipynb) | Previous: [BHaH Project Anatomy](bhah_project_anatomy.ipynb) | Next: [Carpet WaveToy Thorns](carpet_wavetoy_thorns.ipynb)

## Why This Matters

ETLegacy writes Einstein Toolkit thorn metadata and source files. Understanding the generated CCL and schedule files explains how NRPy kernels enter Cactus.

## What You Will See

- Toy gridfunction, parameter, and C-function registration.
- Generated Cactus metadata files.
- Representative generated source files.

## Table of Contents

1. [Helper roles](#Helper-roles)
2. [Create a toy thorn workspace](#Create-a-toy-thorn-workspace)
3. [Register gridfunctions, parameters, and C functions](#Register-gridfunctions,-parameters,-and-C-functions)
4. [Emit Cactus metadata and source files](#Emit-Cactus-metadata-and-source-files)
5. [Inspect generated files](#Inspect-generated-files)

## Helper Roles

The ETLegacy helpers separate thorn generation into small responsibilities:

- `interface_ccl` declares the thorn name, inherited thorns, provided gridfunction groups, and required Cactus functions.
- `param_ccl` exposes selected NRPy code parameters as Cactus parameters.
- `schedule_ccl` writes schedule blocks from registered C-function metadata.
- `make_code_defn` writes registered C functions and lists their source files.
- `zero_rhss`, `MoL_registration`, `Symmetry_registration`, and `boundary_conditions` register common Cactus helper functions.
- `simple_loop` provides compact C loop templates.
- `CodeParameters` emits C code for reading Cactus parameters inside generated functions.

## Create a Toy Thorn Workspace

This notebook writes to a temporary workspace. The thorn is generated for inspection only; building it requires an Einstein Toolkit checkout.

In [1]:
from pathlib import Path
import tempfile

import nrpy

print("nrpy import succeeded")

workspace_manager = tempfile.TemporaryDirectory(prefix="nrpy-etlegacy-")
WORKSPACE = Path(workspace_manager.name).resolve()
PROJECT_DIR = WORKSPACE / "project" / "toy_etlegacy"
THORN_NAME = "ToyETLegacyNRPy"

print("workspace: temporary directory")
print(f"thorn directory: project/toy_etlegacy/{THORN_NAME}")

nrpy import succeeded
workspace: temporary directory
thorn directory: project/toy_etlegacy/ToyETLegacyNRPy


## Register Gridfunctions, Parameters, and C Functions

The toy thorn has one evolved gridfunction and one auxiliary gridfunction. The names are valid scalar base names, so the symmetry helper can classify their parity without ambiguity.

In [2]:
import nrpy.c_function as cfc
import nrpy.grid as gri
import nrpy.params as par
from nrpy.infrastructures.ETLegacy import (
    CodeParameters,
    MoL_registration,
    Symmetry_registration,
    boundary_conditions,
    interface_ccl,
    make_code_defn,
    param_ccl,
    schedule_ccl,
    simple_loop,
    zero_rhss,
)

cfc.CFunction_dict.clear()
gri.glb_gridfcs_dict.clear()

par.set_parval_from_str("Infrastructure", "ETLegacy")

par.register_CodeParameter("CCTK_INT", __name__, "fd_order", 4)
par.register_CodeParameter("CCTK_REAL", __name__, "wave_speed", 1.0)

gri.register_gridfunctions(["psi"], group="EVOL")
gri.register_gridfunctions(["energy"], group="AUX")

print("gridfunction classes:")
for name, gf in sorted(gri.glb_gridfcs_dict.items()):
    print(f"  {name}: {type(gf).__name__}")

body = f"DECLARE_CCTK_ARGUMENTS_{THORN_NAME}_rhs_eval;\nDECLARE_CCTK_PARAMETERS;\n"
body += CodeParameters.read_CodeParameters(
    list_of_tuples__thorn_CodeParameter=[(THORN_NAME, "wave_speed")],
    declare_invdxxs=True,
)
body += simple_loop.simple_loop(
    loop_body=(
        f"{gri.ETLegacyGridFunction.access_gf('psi_rhs')} = "
        f"wave_speed * ({gri.ETLegacyGridFunction.access_gf('energy')} - "
        f"{gri.ETLegacyGridFunction.access_gf('psi')});"
    ),
    loop_region="interior",
)

cfc.register_CFunction(
    subdirectory=THORN_NAME,
    includes=["cctk.h", "cctk_Arguments.h", "cctk_Parameters.h"],
    desc="Evaluate a toy right-hand side.",
    cfunc_type="void",
    name=f"{THORN_NAME}_rhs_eval",
    params="CCTK_ARGUMENTS",
    body=body,
    ET_thorn_name=THORN_NAME,
    ET_schedule_bins_entries=[
        (
            "MoL_CalcRHS",
            """
schedule FUNC_NAME in MoL_CalcRHS
{
  LANG: C
  READS: evol_variables(everywhere)
  READS: aux_variables(everywhere)
  WRITES: evol_variables_rhs(interior)
} "Evaluate toy RHS"
""",
        )
    ],
    ET_current_thorn_CodeParams_used=["wave_speed"],
)

zero_rhss.register_CFunction_zero_rhss(THORN_NAME)
MoL_registration.register_CFunction_MoL_registration(THORN_NAME)
Symmetry_registration.register_CFunction_Symmetry_registration_oldCartGrid3D(THORN_NAME)
boundary_conditions.register_CFunction_specify_Driver_BoundaryConditions(
    thorn_name=THORN_NAME
)

print("registered C functions:")
for name in sorted(cfc.CFunction_dict):
    print(" ", name)

gridfunction classes:
  energy: ETLegacyGridFunction
  psi: ETLegacyGridFunction
registered C functions:
  ToyETLegacyNRPy_MoL_registration
  ToyETLegacyNRPy_Symmetry_registration_oldCartGrid3D
  ToyETLegacyNRPy_rhs_eval
  ToyETLegacyNRPy_specify_Driver_BoundaryConditions
  ToyETLegacyNRPy_zero_rhss


## Emit Cactus Metadata and Source Files

The CCL helpers translate NRPy metadata into Cactus files. `make_code_defn` then writes the registered C functions into the thorn `src/` directory.

In [3]:
import contextlib
import io
import re


def clean_generation_output(text):
    text = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", text)
    text = text.replace(str(WORKSPACE), "temporary workspace")
    text = text.replace(str(PROJECT_DIR), "project/toy_etlegacy")
    return "\n".join(line.rstrip() for line in text.splitlines() if line.strip())


generation_output = io.StringIO()
with contextlib.redirect_stdout(generation_output):
    interface_ccl.construct_interface_ccl(
        project_dir=str(PROJECT_DIR),
        thorn_name=THORN_NAME,
        inherits="Boundary Grid MethodofLines",
        USES_INCLUDEs="USES INCLUDE: Symmetry.h\nUSES INCLUDE: Boundary.h\n",
        is_evol_thorn=True,
    )

    param_ccl.construct_param_ccl(
        project_dir=str(PROJECT_DIR),
        thorn_name=THORN_NAME,
    )

    schedule_ccl.construct_schedule_ccl(
        project_dir=str(PROJECT_DIR),
        thorn_name=THORN_NAME,
        STORAGE="""
STORAGE: evol_variables[3]
STORAGE: evol_variables_rhs[1]
STORAGE: aux_variables[1]
""",
    )

    make_code_defn.output_CFunctions_and_construct_make_code_defn(
        project_dir=str(PROJECT_DIR),
        thorn_name=THORN_NAME,
    )

cleaned_output = clean_generation_output(generation_output.getvalue())
if cleaned_output:
    print(cleaned_output)

required_files = [
    "interface.ccl",
    "param.ccl",
    "schedule.ccl",
    "src/make.code.defn",
]

missing = []
for relative_path in required_files:
    path = PROJECT_DIR / THORN_NAME / relative_path
    print(f"{relative_path}: {path.exists()}")
    if not path.exists():
        missing.append(relative_path)

if missing:
    raise RuntimeError("missing generated files: " + ", ".join(missing))

schedule_text = (PROJECT_DIR / THORN_NAME / "schedule.ccl").read_text(encoding="utf-8")
unexpected_schedule_terms = [
    f"after {THORN_NAME}_RHS",
    f"{THORN_NAME}_specify_NewRad_BoundaryConditions_parameters",
]
present_terms = [term for term in unexpected_schedule_terms if term in schedule_text]
print("boundary schedule check:", not present_terms)
if present_terms:
    raise RuntimeError("unexpected boundary schedule terms: " + ", ".join(present_terms))

Checking temporary workspace/project/toy_etlegacy/ToyETLegacyNRPy/interface.ccl...[written]
Checking temporary workspace/project/toy_etlegacy/ToyETLegacyNRPy/param.ccl...[written]
Checking temporary workspace/project/toy_etlegacy/ToyETLegacyNRPy/schedule.ccl...[written]
Checking temporary workspace/project/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_MoL_registration.c...[written]
Checking temporary workspace/project/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_Symmetry_registration_oldCartGrid3D.c...[written]
Checking temporary workspace/project/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_rhs_eval.c...[written]
Checking temporary workspace/project/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_specify_Driver_BoundaryConditions.c...[written]
Checking temporary workspace/project/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_zero_rhss.c...[written]
Checking temporary workspace/project/toy_etlegacy/ToyETLegacyNRPy/src/make.code.defn...[written]
interface.ccl: True
param.

## Inspect Generated Files

These four files are the core Cactus interface of the generated thorn. They are sufficient to verify ETLegacy wiring without compiling inside the Einstein Toolkit.

In [4]:
for relative_path in required_files:
    path = PROJECT_DIR / THORN_NAME / relative_path
    lines = path.read_text(encoding="utf-8").splitlines()
    print(f"\n{relative_path}")
    print(f"  lines: {len(lines)}")
    for line in lines[:8]:
        print(" ", line)

source_files = sorted(path.name for path in (PROJECT_DIR / THORN_NAME / "src").glob("*.c"))
print("\nC source files:")
for name in source_files:
    print(" ", name)


interface.ccl
  lines: 60
  
  # This interface.ccl file was automatically generated by NRPy.
  #   You are advised against modifying it directly; instead
  #   modify the Python code that generates it.
  
  # With "implements", we give our thorn its unique name.
  implements: ToyETLegacyNRPy
  

param.ccl
  lines: 16
  # This param.ccl file was automatically generated by NRPy.
  #   You are advised against modifying it directly; instead
  #   modify the Python code that generates it.
  
  
  restricted:
  CCTK_INT fd_order "(see NRPy for parameter definition)"
  {

schedule.ccl
  lines: 55
  # This schedule.ccl file was automatically generated by NRPy.
  #   You are advised against modifying it directly; instead
  #   modify the Python code that generates it.
  
  ##################################################
  # Step 0: Allocate memory for gridfunctions, using the STORAGE: keyword.
  
  STORAGE: evol_variables[3]

src/make.code.defn
  lines: 9
  # make.code.defn file for thorn 

The ETLegacy generation path is complete at this point. `interface.ccl`, `param.ccl`, and `schedule.ccl` describe the thorn to Cactus; `src/make.code.defn` tells the Einstein Toolkit build system which generated C files to compile.

## Where This Leads

- [BHaH Project Anatomy](bhah_project_anatomy.ipynb) reviews the prerequisite step.
- [Carpet WaveToy Thorns](carpet_wavetoy_thorns.ipynb) continues the tutorial path.
- [Index](../index.ipynb) returns to the full tutorial spine.